# Wearable daily fanout

Scheduled daily pull-based ingestion across every registered cloud-API connector.
Reads the `wearable_credentials` Lakebase table, groups by provider, and dispatches
to each provider's Python `ConnectorProtocol` implementation under
`providers/<name>/pull/`.

Both the webhook path (inside the AppKit app) and this pull path write the **same
bronze row shape** to `wearables_zerobus` — silver pipelines don't need to distinguish.

**Prerequisites:**
- `zeroBus/dbxW_zerobus_infra` bundle deployed (bronze table + ZeroBus SPN provisioned).
- AppKit app deployed at least once (Lakebase schema + `wearable_credentials` exist).
- At least one user has linked a provider through the AppKit Connections UI — or the
  dev fallback `providers/garmin/scripts/upload_garmin_tokens.sh` has been run.

In [ ]:
%pip install garminconnect>=0.2.0 databricks-zerobus-ingest-sdk>=1.0.0 psycopg[binary]>=3.2.0

In [ ]:
DEFAULT_SECRET_SCOPE = "dbxw_zerobus_credentials"
DEFAULT_BRONZE_TABLE = ""  # filled from secret scope at runtime
DEFAULT_PROVIDERS = "garmin"  # comma-separated list

try:
    dbutils.widgets.text("secret_scope", DEFAULT_SECRET_SCOPE, "Secret Scope")
    dbutils.widgets.text("providers", DEFAULT_PROVIDERS, "Providers (comma-separated, blank=all)")
    dbutils.widgets.text("target_date", "", "Target Date (YYYY-MM-DD, blank=yesterday)")
    dbutils.widgets.text("delay_seconds", "2", "Delay between users (seconds)")

    secret_scope = dbutils.widgets.get("secret_scope")
    providers_arg = dbutils.widgets.get("providers").strip()
    target_date_str = dbutils.widgets.get("target_date").strip()
    delay_seconds = float(dbutils.widgets.get("delay_seconds"))
except Exception:
    secret_scope = DEFAULT_SECRET_SCOPE
    providers_arg = DEFAULT_PROVIDERS
    target_date_str = ""
    delay_seconds = 2.0

print(f"Secret scope:  {secret_scope}")
print(f"Providers:     {providers_arg or '(all)'}")
print(f"Target date:   {target_date_str or 'yesterday'}")
print(f"Delay seconds: {delay_seconds}")

## Determine target date

In [ ]:
from datetime import date, timedelta

if target_date_str:
    target_date = date.fromisoformat(target_date_str)
else:
    target_date = date.today() - timedelta(days=1)

from providers.common.connector_protocol import DateRange
range_ = DateRange(start=target_date, end=target_date)
print(f"Target date: {target_date}")

## Wire Lakebase env vars and build the credential store

The AppKit `wearable-core` plugin writes OAuth tokens into the Lakebase
`wearable_credentials` table. This notebook reads them via the same
envelope-encryption framing (AES-256-GCM) using the shared signing key.

In [ ]:
import os

os.environ["PGHOST"] = dbutils.secrets.get(scope=secret_scope, key="lakebase_pghost")
os.environ["PGDATABASE"] = dbutils.secrets.get(scope=secret_scope, key="lakebase_pgdatabase")
os.environ["PGUSER"] = dbutils.secrets.get(scope=secret_scope, key="lakebase_pguser")
os.environ["PGPASSWORD"] = dbutils.secrets.get(scope=secret_scope, key="lakebase_pgpassword")
os.environ.setdefault("PGPORT", "5432")
os.environ.setdefault("PGSSLMODE", "require")
os.environ["LAKEBASE_SIGNING_KEY"] = dbutils.secrets.get(scope=secret_scope, key="wearable_core_signing_key")

from providers.common.credential_store import LakebaseCredentialStore
store = LakebaseCredentialStore.from_env()

## Import provider pull implementations

Each provider's `pull/__init__.py` self-registers at import time.
Uncomment provider imports as they are implemented.

In [ ]:
import providers.garmin.pull  # noqa: F401
# import providers.fitbit.pull  # noqa: F401
# import providers.whoop.pull   # noqa: F401
# import providers.oura.pull    # noqa: F401
# import providers.withings.pull  # noqa: F401
# import providers.strava.pull  # noqa: F401

from providers.common.connector_protocol import (
    get_connector, list_registered, ProviderRateLimitError,
)

if providers_arg:
    providers_list = [p.strip() for p in providers_arg.split(",") if p.strip()]
else:
    providers_list = list_registered()
print("Will process providers:", providers_list)

## ZeroBus writer

Reuses the infra-bundle-provisioned ZeroBus SPN credentials. Same stream
is used for every user across every provider for the duration of this run.

In [ ]:
from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties
from zerobus.sdk.sync import ZerobusSdk

workspace_url = dbutils.secrets.get(scope=secret_scope, key="workspace_url")
zerobus_endpoint = dbutils.secrets.get(scope=secret_scope, key="zerobus_endpoint")
client_id = dbutils.secrets.get(scope=secret_scope, key="client_id")
client_secret = dbutils.secrets.get(scope=secret_scope, key="client_secret")
bronze_table = dbutils.secrets.get(scope=secret_scope, key="target_table_name")

sdk = ZerobusSdk(zerobus_endpoint, workspace_url)
table_props = TableProperties(bronze_table)
options = StreamConfigurationOptions(record_type=RecordType.JSON, max_inflight_requests=50)
stream = sdk.create_stream(client_id, client_secret, table_props, options)
print(f"ZeroBus stream opened -> {bronze_table}")

## Fanout loop

Per-provider sliding-window rate limiting plus circuit breaker on repeated 429s.

In [ ]:
import time
import uuid
from datetime import datetime, timezone

CIRCUIT_BREAKER_THRESHOLD = 3
run_id = str(uuid.uuid4())
started_at = datetime.now(timezone.utc)

summary = {"users": 0, "rows": 0, "errors": 0, "rate_limited": 0, "skipped": 0}

for provider in providers_list:
    try:
        connector = get_connector(provider)
    except KeyError as e:
        print(f"[SKIP] {e}")
        summary["skipped"] += 1
        continue
    
    consecutive_rate_limits = 0
    for enrolled in store.list_enrolled(provider=provider):
        if consecutive_rate_limits >= CIRCUIT_BREAKER_THRESHOLD:
            print(f"[CIRCUIT BREAK] {provider} — tripped after {consecutive_rate_limits} rate-limit errors")
            summary["skipped"] += 1
            break
        summary["users"] += 1
        try:
            rows = connector.pull_batch(enrolled.user_id, enrolled.provider_user_id, range_)
        except ProviderRateLimitError:
            consecutive_rate_limits += 1
            summary["rate_limited"] += 1
            print(f"[429] {provider}/{enrolled.user_id}")
            time.sleep(60 * 2 ** (consecutive_rate_limits - 1))
            continue
        except Exception as exc:
            summary["errors"] += 1
            print(f"[ERROR] {provider}/{enrolled.user_id}: {exc}")
            continue
        consecutive_rate_limits = 0
        offsets = []
        for row in rows:
            offsets.append(stream.ingest_record_offset({
                "record_id": row.record_id,
                "ingested_at": row.ingested_at,
                "body": row.body,
                "headers": row.headers,
                "record_type": row.record_type,
            }))
        summary["rows"] += len(rows)
        time.sleep(delay_seconds)
    
    if offsets:
        stream.wait_for_offset(offsets[-1])

stream.close()
finished_at = datetime.now(timezone.utc)
print(f"Run {run_id} complete in {(finished_at - started_at).total_seconds():.1f}s: {summary}")

## Observability

The connector's `pull_batch` is expected to write individual `wearable_sync_runs`
rows (status ok / error / rate_limited) per user. This notebook could also write
one aggregate row per run here if desired.